In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
import torch.optim as optim
from deepsea import DeepSEA
from data import X, Y
import torch.nn as nn
import torch
import math

X_tensor = torch.from_numpy(X).float().permute(0, 2, 1)
Y_tensor = torch.from_numpy(Y).float()

# positional split: first 80% train, last 20% val
split = int(0.8 * len(X_tensor))
X_train, X_val = X_tensor[:split], X_tensor[split:]
Y_train, Y_val = Y_tensor[:split], Y_tensor[split:]

train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=128, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   Y_val),   batch_size=512, shuffle=False)

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = DeepSEA(n_outputs=5).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

marks = ['H3k4me1', 'H3k4me3', 'H3k27ac', 'DNase', 'CTCF']

print(f"Device: {device}")
print(f"Train samples: {len(X_train):,} | Val samples: {len(X_val):,}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print("-" * 80)

best_val_loss = math.inf
log_interval  = max(1, len(train_loader) // 5)  # print ~5 times per epoch

for epoch in range(30):
    # --- train ---
    model.train()
    train_loss   = 0.0
    batches_seen = 0
    for batch_idx, (xb, yb) in enumerate(train_loader):
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        train_loss   += loss.item() * len(xb)
        batches_seen += 1

        if (batch_idx + 1) % log_interval == 0:
            running_loss = train_loss / (batches_seen * train_loader.batch_size)
            pct = 100.0 * (batch_idx + 1) / len(train_loader)
            print(f"  Epoch {epoch+1:>2d} [{pct:5.1f}%] batch {batch_idx+1}/{len(train_loader)} "
                  f"| running loss {running_loss:.4f}")

    train_loss /= len(X_train)

    # --- validate ---
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            all_logits.append(model(xb.to(device)).cpu())
            all_labels.append(yb)
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)

    val_loss = criterion(logits, labels).item()
    probs    = torch.sigmoid(logits).numpy()

    aurocs = []
    for i in range(5):
        try:
            aurocs.append(roc_auc_score(labels[:, i], probs[:, i]))
        except ValueError:
            aurocs.append(float('nan'))   # fires if val split has no positives for a mark

    mean_auroc = sum(a for a in aurocs if not math.isnan(a)) / sum(1 for a in aurocs if not math.isnan(a))

    print(f"Epoch {epoch+1:>2d} | train {train_loss:.4f} | val {val_loss:.4f} | "
          f"mean AUROC {mean_auroc:.3f} | "
          + " ".join(f"{m}={a:.3f}" for m, a in zip(marks, aurocs)))

    # --- checkpoint ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch':      epoch + 1,
            'state_dict': model.state_dict(),
            'optimizer':  optimizer.state_dict(),
            'val_loss':   val_loss,
            'aurocs':     dict(zip(marks, aurocs)),
        }, 'best_model.pt')
        print(f"  --> Saved checkpoint (val loss improved to {val_loss:.4f})")

print("-" * 80)
print(f"Training complete. Best val loss: {best_val_loss:.4f} | checkpoint: best_model.pt")

48129895
NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN
read first file
read second file
read third file
read fourth file
read all files
X shape: (27558, 1000, 4) Y shape: (27558, 5)
positives per mark:
  H3k4me1    20596  (74.7%)
  H3k4me3    15953  (57.9%)
  H3k27ac     9091  (33.0%)
  DNase       2377  (8.6%)
  CTCF         754  (2.7%)
marks per window: min 1 mean 1.77
fraction N bases: 0.0
Device: cpu
Train samples: 22,046 | Val samples: 5,512
Train batches: 173 | Val batches: 11
--------------------------------------------------------------------------------
  Epoch  1 [ 19.7%] batch 34/173 | running loss 0.9414
  Epoch  1 [ 39.3%] batch 68/173 | running loss 0.6994
  Epoch  1 [ 59.0%] batch 102/173 | running loss 0.6197
  Epoch  1 [ 78.6%] batch 136/173 | running loss 0.5794
  Epoch  1 [ 98.3%] batch 170/173 | running loss 0.5542
Epoch  1 | train 0.5527 | val 0.4679 | mean AUROC 0.582 | H3k4me1=0.544 H3k4me3=0.506 H3k27ac=0.59